In [ ]:
!pip -q install --upgrade xee geemap hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.1/477.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='vothanhhuy', opt_url="https://earthengine-highvolume.googleapis.com") # Replace 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID

In [ ]:
import geemap
m = geemap.Map()
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
if m.draw_last_feature:
    roi = m.draw_last_feature.geometry()
    print('Đã lấy được hình học của đối tượng bạn đã vẽ:')
    print(roi)
else:
    print('Bạn chưa vẽ bất kỳ đối tượng nào trên bản đồ. Vui lòng vẽ một đối tượng trên bản đồ Geemap trước khi chạy ô này.')

Đã lấy được hình học của đối tượng bạn đã vẽ:
ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          108.237305,
                          12.806445
                        ],
                        [
                          108.237305,
                          13.576581
                        ],
                        [
                          109.79187,
                          13.576581
                        ],
                        [
                          109.79187,
                          1

In [ ]:
roi = m.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          108.237305,
                          12.806445
                        ],
                        [
                          108.237305,
                          13.576581
                        ],
                        [
                          109.79187,
                          13.576581
                        ],
                        [
                          109.79187,
                          12.806445
                        ],
                        [
                          108.237305,
                          12.806445
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [ ]:
viirs = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate('2020-01-01', '2021-01-01')
    .select('EVI')
)
viirs.size().getInfo()

46

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    viirs,
    engine='ee',
    crs='EPSG:4326',
    geometry=roi,
    scale=0.1  # theo code mẫu bạn đưa
)

ds = ds.sortby('time') * 1
ds

<xarray.Dataset> Size: 24kB
Dimensions:  (time: 46, lon: 16, lat: 8)
Coordinates:
  * time     (time) datetime64[ns] 368B 2020-01-01 2020-01-09 ... 2020-12-26
  * lon      (lon) float64 128B 108.3 108.4 108.5 108.6 ... 109.6 109.7 109.8
  * lat      (lat) float64 64B 12.86 12.96 13.06 13.16 13.26 13.36 13.46 13.56
Data variables:
    EVI      (time, lon, lat) float32 24kB 0.4014 0.4038 0.401 ... nan nan nan
Attributes:
    crs:      EPSG:4326

In [ ]:
ds_mean = ds.mean(dim='time')
ds_mean

<xarray.Dataset> Size: 704B
Dimensions:  (lon: 16, lat: 8)
Coordinates:
  * lon      (lon) float64 128B 108.3 108.4 108.5 108.6 ... 109.6 109.7 109.8
  * lat      (lat) float64 64B 12.86 12.96 13.06 13.16 13.26 13.36 13.46 13.56
Data variables:
    EVI      (lon, lat) float32 512B 0.4125 0.427 0.4393 0.4501 ... nan nan nan
Attributes:
    crs:      EPSG:4326

In [ ]:
import hvplot.xarray  # quan trọng

ds_mean.EVI.hvplot(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    title='Average EVI - 2020'
)

:Image   [lon,lat]   (EVI)

In [ ]:
ds_mean.EVI.hvplot.quadmesh(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True
)

:QuadMesh   [lon,lat]   (EVI)

In [ ]:
ds_mean.EVI.hvplot.contour(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    levels=10
)

:Contours   [lon,lat]   (EVI)

In [ ]:
ds_mean.EVI.hvplot.contour(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    levels=20
)

:Contours   [lon,lat]   (EVI)

In [ ]:
ds_mean.EVI.hvplot.contourf(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    levels=20
)

:Polygons   [lon,lat]   (EVI)

In [ ]:
ds_mean.EVI.hvplot.contourf(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    levels=40
)

:Polygons   [lon,lat]   (EVI)

In [ ]:
ds.EVI.hvplot(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    groupby='time',
    widget_location='bottom',
    widget_type='scrubber'
)

Column
    [0] HoloViews(DynamicMap, height=500, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=700)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=45, width=550)

In [ ]:
ds.EVI.hvplot(
    x='lon', y='lat',
    height=500,
    cmap='RdYlGn',
    robust=True,
    groupby='time',
    widget_location='bottom',
    widget_type='scrubber' # Đã sửa từ 'select' thành 'scrubber'
)

Column
    [0] HoloViews(DynamicMap, height=500, sizing_mode='fixed', widget_location='bottom', widget_type='scrubber', width=700)
    [1] WidgetBox(align=('center', 'end'))
        [0] Player(end=45, width=550)

In [ ]:
viirs2 = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate('2020-01-01', '2021-01-01')
    .select('NDVI')  # nếu có
)